In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR = Path('../data')

In [14]:
baseline = pd.read_csv(DATA_DIR / 'TX_upgrade0.csv')
print(f'Rows: {len(baseline):,}  |  Columns: {len(baseline.columns)}')
baseline.head()

Rows: 42,845  |  Columns: 771


/var/folders/n_/rbh3_bsn3h9cv6rx2h1bftm80000gn/T/ipykernel_88427/3255948953.py:1: DtypeWarning: Columns (144) have mixed types. Specify dtype option on import or set low_memory=False.
  baseline = pd.read_csv(DATA_DIR / 'TX_upgrade0.csv')


,bldg_id,completed_status,upgrade,in.upgrade_name,weight,applicability,in.sqft..ft2,in.representative_income,in.county_name,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_sub_cz_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_unavailable_days,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_metro_status,in.custom_state,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,in.door_area,in.doors,in.duct_leakage_and_insulation,in.duct_location,in.eaves,in.electric_panel_service_rating..a,in.electric_panel_service_rating_bin..a,in.electric_vehicle_battery,in.electric_vehicle_charge_at_home,in.electric_vehicle_charger,in.electric_vehicle_miles_traveled,in.electric_vehicle_outlet_access,in.electric_vehicle_ownership,in.energystar_climate_zone_2023,in.federal_poverty_level,in.generation_and_emissions_assessment_region,in.geometry_attic_type,in.geometry_building_horizontal_location_mf,in.geometry_building_horizontal_location_sfa,in.geometry_building_level_mf,in.geometry_building_number_units_mf,in.geometry_building_number_units_sfa,in.geometry_building_type_acs,in.geometry_building_type_height,in.geometry_building_type_recs,in.geometry_floor_area,in.geometry_floor_area_bin,in.geometry_foundation_type,in.geometry_garage,in.geometry_space_combination,in.geometry_stories,in.geometry_stories_low_rise,in.geometry_story_bin,in.geometry_wall_exterior_finish,in.geometry_wall_type,in.ground_thermal_conductivity,in.has_pv,in.heating_fuel,in.heating_unavailable_days,in.heating_setpoint,in.heating_setpoint_has_offset,in.heating_setpoint_offset_magnitude,in.heating_setpoint_offset_period,in.holiday_lighting,in.hot_water_distribution,in.hot_water_fixtures,in.household_has_tribal_persons,in.hvac_cooling_autosizing_factor,in.hvac_cooling_efficiency,in.hvac_cooling_partial_space_conditioning,in.hvac_cooling_type,in.hvac_has_ducts,in.hvac_has_shared_system,in.hvac_has_zonal_electric_heating,in.hvac_heating_autosizing_factor,in.hvac_heating_efficiency,in.hvac_heating_type,in.hvac_heating_type_and_fuel,in.hvac_secondary_heating_efficiency,in.hvac_secondary_heating_fuel,in.hvac_secondary_heating_partial_space_conditioning,in.hvac_secondary_heating_type,in.hvac_shared_efficiencies,in.hvac_system_is_faulted,in.hvac_system_is_scaled,in.hvac_system_single_speed_ac_airflow,in.hvac_system_single_speed_ac_charge,in.hvac_system_single_speed_ashp_airflow,in.hvac_system_single_speed_ashp_charge,in.income,in.income_recs_2015,in.income_recs_2020,in.infiltration,in.insulation_ceiling,in.insulation_floor,in.insulation_foundation_wall,in.insulation_rim_joist,in.insulation_roof,in.insulation_slab,in.insulation_wall,in.interior_shading,in.iso_rto_region,in.lighting,in.lighting_interior_use,in.lighting_other_use,in.location_region,in.mechanical_ventilation,in.metropolitan_and_micropolitan_statistical_area,in.misc_extra_refrigerator,in.misc_freezer,in.misc_gas_fireplace,in.misc_gas_grill,in.misc_gas_lighting,in.misc_hot_tub_spa,in.misc_pool,in.misc_pool_heater,in.misc_pool_pump,in.misc_well_pump,in.natural_ventilation,in.neighbors,in.occupants,in.orientation,in.overhangs,in.plug_load_diversity,in.plug_loads,in.puma,in.puma_metro_status,in.pv_orientation,in.pv_system_size,in.radiant_barrier,in.range_spot_vent_hour,in.reeds_balancing_area,in.refrigerator,in.refrigerator_usage_level,in.roof_material,in.state,in.state_metro_median_income,in.tenure,in.usage_level,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_i

In [15]:
SF_TYPES = ['Single-Family Attached', 'Single-Family Detached']
SIZE_ORDER = ['<100', '100', '101-124', '125', '126-199', '200', '201+']

sf = baseline[baseline['in.geometry_building_type_acs'].isin(SF_TYPES)].copy()
print(f'Single-family homes: {len(sf):,} records ({sf["weight"].sum():,.0f} weighted units)')

Single-family homes: 28,899 records (7,337,562 weighted units)


In [12]:
# Panel size distribution — all homes and single-family
for label, subset in [('All homes', baseline), ('Single-family only', sf)]:
    total = subset['weight'].sum()
    result = subset.groupby('in.electric_panel_service_rating_bin..a')['weight'].sum()
    print(f'=== {label} (n={total:,.0f} weighted units) ===')
    for size in SIZE_ORDER:
        if size in result:
            v = result[size]
            print(f'  {size}A: {v:,.0f} ({v/total*100:.1f}%)')
    small = subset[subset['in.electric_panel_service_rating_bin..a'].isin(['<100', '100', '101-124', '125'])]
    print(f'  <= 125A total: {small["weight"].sum():,.0f} ({small["weight"].sum()/total*100:.1f}%)')
    print()

=== All homes (n=10,878,503 weighted units) ===
  <100A: 260,759 (2.4%)
  100A: 2,054,081 (18.9%)
  101-124A: 66,015 (0.6%)
  125A: 2,000,253 (18.4%)
  126-199A: 504,253 (4.6%)
  200A: 5,728,321 (52.7%)
  201+A: 264,822 (2.4%)
  <= 125A total: 4,381,108 (40.3%)

=== Single-family only (n=7,337,562 weighted units) ===
  <100A: 108,417 (1.5%)
  100A: 1,550,590 (21.1%)
  101-124A: 41,894 (0.6%)
  125A: 453,980 (6.2%)
  126-199A: 287,673 (3.9%)
  200A: 4,690,363 (63.9%)
  201+A: 204,646 (2.8%)
  <= 125A total: 2,154,880 (29.4%)

